# ML model results

Reads **all experimental runs** from `saved/ml_results.parquet` (written by `02_ml_models_fit.ipynb`).

Six models × four feature tables:

| Dataset | Features |
|---|---|
| Baseline | original columns, no `uid` / `uid2` / `DT_*` |
| Feature Engineering | plus `uid`, `uid2`, `DT_*` |
| Reduced Baseline | Table 3 filters, no `uid` / `uid2` / `DT_*` |
| Reduced Feature Engineering | reduced plus surviving `uid` / `uid2` / `DT_*` |

Models: logistic regression, decision tree, random forest, LightGBM, XGBoost, CatBoost.

Optuna `BestParams` are used to refit on **all labeled train rows** and score `merged_test.parquet`. Submissions go to `results/{ModelType}_{Dataset}_submission.csv` with columns `TransactionID`, `IsFraud` (fraud probability).


In [1]:
import warnings
import numpy as np
import pandas as pd
import json
import gc
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    balanced_accuracy_score,
    matthews_corrcoef,
    classification_report,
)
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")
pd.options.display.precision = 4

print("ML model results")


ML model results


In [2]:
# Environment & Paths Setup

try:
    import google.colab
    IS_COLAB = True
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    ROOT = Path("/content/drive/MyDrive/minor-thesis")
else:
    ROOT = Path.cwd()

DATASET_PATH = ROOT / "dataset"
SAVED_PATH = ROOT / "saved"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Environment: {"Google Colab" if IS_COLAB else "Local"}')
print(f"Dataset path: {DATASET_PATH}")
print(f"Results table: {SAVED_PATH / 'ml_results.parquet'}")
print(f"Submission dir: {RESULTS_DIR}")


Environment: Local
Dataset path: d:\source\RMIT\master-of-ai-new\2026-semester-02\minor-thesis\dataset
Results table: d:\source\RMIT\master-of-ai-new\2026-semester-02\minor-thesis\saved\ml_results.parquet
Submission dir: d:\source\RMIT\master-of-ai-new\2026-semester-02\minor-thesis\results


## Experiment results

One table of every trained run in `saved/ml_results.parquet` (6 models × 4 datasets). Re-run any missing family in `02_ml_models_fit.ipynb`.


In [3]:
from IPython.display import display

ml_results_path = SAVED_PATH / "ml_results.parquet"
all_results = pd.read_parquet(ml_results_path)
all_results = all_results[
    ~all_results["Model"].astype(str).str.startswith(("SVM -", "KNN -"))
]

MODEL_ORDER = [
    "LogisticRegression",
    "DecisionTree",
    "RandomForest",
    "LightGBM",
    "XGBoost",
    "CatBoost",
]
DATASET_ORDER = [
    "Baseline",
    "Feature Engineering",
    "Reduced Baseline",
    "Reduced Feature Engineering",
]
NAME_PREFIX = {
    "LogisticRegression": "Logistic Regression",
    "DecisionTree": "Decision Tree",
    "RandomForest": "RF",
    "LightGBM": "LightGBM",
    "XGBoost": "XGBoost",
    "CatBoost": "CatBoost",
}

if "Dataset" not in all_results.columns:
    all_results["Dataset"] = all_results["Model"].astype(str).str.split(" - ", n=1).str[1]

expected = {
    f"{prefix} - {dataset}"
    for prefix in NAME_PREFIX.values()
    for dataset in DATASET_ORDER
}
missing = sorted(expected - set(all_results["Model"].astype(str)))

show_cols = [
    "ModelType",
    "Dataset",
    "Features",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC",
    "TuneROC-AUC",
    "PR-AUC",
    "Balanced Accuracy",
    "MCC",
    "TP",
    "FP",
    "FN",
    "TN",
    "BestParams",
]
show_cols = [c for c in show_cols if c in all_results.columns]

results_table = all_results.copy()
results_table["ModelType"] = pd.Categorical(
    results_table["ModelType"], categories=MODEL_ORDER, ordered=True
)
results_table["Dataset"] = pd.Categorical(
    results_table["Dataset"], categories=DATASET_ORDER, ordered=True
)
results_table = (
    results_table.sort_values(["ModelType", "Dataset"])[show_cols].reset_index(drop=True)
)

print(
    f"{ml_results_path.name}: {len(results_table)} trained runs "
    f"(expected 24 = 6 models x 4 datasets)"
)
if missing:
    print("Not trained yet:")
    for name in missing:
        print(f"  {name}")
print()
display(results_table)


ml_results.parquet: 24 trained runs (expected 24 = 6 models x 4 datasets)



,ModelType,Dataset,Features,Accuracy,Precision,Recall,F1,ROC-AUC,TuneROC-AUC,PR-AUC,Balanced Accuracy,MCC,TP,FP,FN,TN,BestParams
0,LogisticRegression,Baseline,432,0.7058,0.0878,0.8039,0.1583,0.8335,0.8538,0.2067,0.7531,0.1986,3267,33946,797,80098,"{""C"": 1.9942666309136752}"
1,LogisticRegression,Feature Engineering,439,0.7091,0.0885,0.8022,0.1595,0.8334,0.8517,0.2094,0.7540,0.1999,3260,33556,804,80488,"{""C"": 6.051195987836167}"
2,LogisticRegression,Reduced Baseline,342,0.6881,0.0834,0.8076,0.1513,0.8248,0.8486,0.1771,0.7457,0.1901,3282,36051,782,77993,"{""C"": 9.177237615941623}"
3,LogisticRegression,Reduced Feature Engineering,348,0.6976,0.0850,0.7975,0.1536,0.8248,0.8462,0.1802,0.7457,0.1916,3241,34897,823,79147,"{""C"": 2.9154431891537547}"
4,DecisionTree,Baseline,432,0.8150,0.1167,0.6663,0.1986,0.8251,0.8385,0.3310,0.7433,0.2233,2708,20493,1356,93551,"{""max_depth"": 8, ""min_samples_leaf"": 43, ""min_..."
5,DecisionTree,Feature Engineering,439,0.8181,0.1198,0.6754,0.2035,0.7965,0.8451,0.3368,0.7493,0.2298,2745,20170,1319,93874,"{""max_depth"": 13, ""min_samples_leaf"": 49, ""min..."
6,DecisionTree,Reduced Baseline,342,0.8237,0.1210,0.6580,0.2044,0.8221,0.8362,0.3320,0.7438,0.2279,2674,19428,1390,94616,"{""max_depth"": 9, ""min_samples_leaf"": 64, ""min_..."
7,DecisionTree,Reduced Feature Engineering,348,0.8306,0.1303,0.6912,0.2193,0.8307,0.8482,0.3506,0.7634,0.2486,2809,18750,1255,95294,"{""max_depth"": 8, ""min_samples_leaf"": 26, ""min_..."
8,RandomForest,Baseline,432,0.9295,0.2690,0.6097,0.3733,0.8959,0.9008,0.4875,0.7753,0.3743,2478,6735,1586,107309,"{""n_estimators"": 250, ""max_depth"": 25, ""min_sa..."
9,RandomForest,Feature Engineering,439,0.9305,0.2710,0.6031,0.3740,0.8961,0.9001,0.4885,0.7726,0.3738,2451,6592,1613,107452,"{""n_estimators"": 250, ""max_depth"": 25, ""min_sa..."


## Top 20 feature importances

Each training run stores its top 20 features (with `%` of total model importance) in `Top20Importances`. Re-run those experiments in `02_ml_models_fit.ipynb` if the column is missing.


In [4]:
def parse_top20(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return None
        return pd.DataFrame(json.loads(text))
    if isinstance(value, (list, tuple)):
        return pd.DataFrame(value)
    return None


def show_top20(model_name):
    rows = all_results[all_results["Model"] == model_name]
    if rows.empty:
        print(f"{model_name}: not in ml_results.parquet")
        return
    row = rows.iloc[0]
    if "Top20Importances" not in all_results.columns:
        print("Top20Importances column missing — re-run 02_ml_models.ipynb")
        return
    table = parse_top20(row["Top20Importances"])
    if table is None or table.empty:
        print(f"{model_name}: no top-20 saved yet — re-run that experiment in 02_ml_models.ipynb")
        return
    table = table.rename(
        columns={
            "rank": "Rank",
            "feature": "Feature",
            "importance": "Importance",
            "importance_pct": "Importance %",
        }
    )
    print(
        f"\n{model_name}  |  {int(row['Features'])} features  |  "
        f"ROC-AUC {row['ROC-AUC']:.4f}  |  F1 {row['F1']:.4f}"
    )
    display(table)


if "Top20Importances" not in all_results.columns:
    print("Top20Importances is not in ml_results.parquet yet.")
    print("Re-run the experiment cells in 02_ml_models.ipynb, then re-load this notebook.")
else:
    n_saved = all_results["Top20Importances"].notna().sum()
    print(f"Runs with top-20 importances: {n_saved} / {len(all_results)}")

    highlight = [
        "Logistic Regression - Baseline",
        "Decision Tree - Baseline",
        "RF - Baseline",
        "LightGBM - Baseline",
        "XGBoost - Baseline",
        "CatBoost - Baseline",
        "RF - Feature Engineering",
        "LightGBM - Feature Engineering",
        "XGBoost - Feature Engineering",
        "CatBoost - Feature Engineering",
        "RF - Reduced Baseline",
        "LightGBM - Reduced Baseline",
        "XGBoost - Reduced Baseline",
        "CatBoost - Reduced Baseline",
        "RF - Reduced Feature Engineering",
        "LightGBM - Reduced Feature Engineering",
        "XGBoost - Reduced Feature Engineering",
        "CatBoost - Reduced Feature Engineering",
    ]
    for name in highlight:
        show_top20(name)

    print("\n===== BEST ROC-AUC PER MODEL FAMILY =====")
    for family in MODEL_ORDER:
        frame = all_results[all_results["ModelType"] == family]
        if frame.empty:
            continue
        best_name = frame.sort_values("ROC-AUC", ascending=False).iloc[0]["Model"]
        print(f"\n{family} best: {best_name}")
        show_top20(best_name)


Runs with top-20 importances: 24 / 24

Logistic Regression - Baseline  |  432 features  |  ROC-AUC 0.8335  |  F1 0.1583


,Rank,Feature,Importance,Importance %
0,1,C11,9.4344,1.7884
1,2,V283,8.4487,1.6015
2,3,C14,8.1261,1.5404
3,4,V300,7.6893,1.4576
4,5,V218,7.1620,1.3576
5,6,V258,6.8203,1.2928
6,7,V87,6.7427,1.2781
7,8,V45,6.6559,1.2617
8,9,C12,6.3649,1.2065
9,10,V123,6.0615,1.1490



Decision Tree - Baseline  |  432 features  |  ROC-AUC 0.8251  |  F1 0.1986


,Rank,Feature,Importance,Importance %
0,1,V218,0.3303,33.0275
1,2,V102,0.1359,13.5950
2,3,C8,0.1086,10.8614
3,4,card6,0.0621,6.2078
4,5,V29,0.0462,4.6187
5,6,V334,0.0315,3.1525
6,7,V306,0.0287,2.8709
7,8,V69,0.0263,2.6262
8,9,V190,0.0214,2.1443
9,10,C14,0.0175,1.7547


RF - Baseline: no top-20 saved yet — re-run that experiment in 02_ml_models.ipynb

LightGBM - Baseline  |  432 features  |  ROC-AUC 0.9109  |  F1 0.4373


...

,Rank,Feature,Importance,Importance %
0,1,card1,2968.0,4.5420
1,2,TransactionDT,2827.0,4.3262
2,3,TransactionAmt,2792.0,4.2726
3,4,card2,2606.0,3.9880
4,5,uid,2466.0,3.7738
5,6,addr1,2408.0,3.6850
6,7,uid2,1752.0,2.6811
7,8,C13,1472.0,2.2526
8,9,D15,1359.0,2.0797
9,10,DT_day,1353.0,2.0705



XGBoost best: XGBoost - Feature Engineering

XGBoost - Feature Engineering  |  439 features  |  ROC-AUC 0.9196  |  F1 0.5813


,Rank,Feature,Importance,Importance %
0,1,V258,0.2107,21.0716
1,2,V257,0.0661,6.6074
2,3,V70,0.0601,6.0129
3,4,V91,0.0363,3.6341
4,5,V294,0.0363,3.6289
5,6,V201,0.0106,1.0583
6,7,V187,0.0082,0.8200
7,8,C14,0.0079,0.7881
8,9,C8,0.0078,0.7767
9,10,V264,0.0077,0.7727



CatBoost best: CatBoost - Reduced Feature Engineering

CatBoost - Reduced Feature Engineering  |  348 features  |  ROC-AUC 0.9095  |  F1 0.3409


,Rank,Feature,Importance,Importance %
0,1,C1,8.4052,8.4052
1,2,card2,4.5254,4.5254
2,3,C13,3.5556,3.5556
3,4,addr1,3.5401,3.5401
4,5,C14,3.4880,3.4880
5,6,TransactionAmt,3.2069,3.2069
6,7,uid,3.1911,3.1911
7,8,card1,2.9155,2.9155
8,9,D2,2.8081,2.8081
9,10,P_emaildomain,2.3440,2.3440


## Imbalanced-data read

Accuracy is misleading at a 3.5% fraud rate. Rank the four-dataset runs by ROC-AUC, PR-AUC, F1, and recall.


In [5]:
print("=" * 130)
print("IMBALANCED DATASET ANALYSIS - FOR FRAUD DETECTION")
print("Focus: ROC-AUC, PR-AUC, F1, Recall (NOT Accuracy - accuracy is misleading!)")
print("=" * 130)

print("\n1. KEY METRICS FOR IMBALANCED DATA")
print("-" * 130)
print("""
For fraud detection (imbalanced dataset):

ROC-AUC (0.85-0.95 good)
  - Threshold-independent, handles class imbalance naturally
  - Main metric for comparing models

PR-AUC (Precision-Recall)
  - Especially important when fraud is rare
  - Shows trade-off: catch more frauds vs avoid false alerts

F1 Score
  - Harmonic mean of Precision & Recall
  - Balances both metrics

Recall (Sensitivity)
  - % of actual frauds caught
  - Missing a fraud = business loss

Precision
  - % of fraud alerts that are actually fraud
  - False alert = customer friction

Confusion Matrix (TN, FP, FN, TP)
  - FN (False Negatives) = frauds we missed - WORST outcome
  - FP (False Positives) = customers falsely flagged - customer friction
  - TP (True Positives) = frauds caught - GOOD
  - TN (True Negatives) = legitimate txns correctly allowed - GOOD

Accuracy - MISLEADING for imbalanced data!
  Example: 99% legitimates, 1% fraud
  Model that predicts "always legitimate" = 99% accuracy but CATCHES ZERO FRAUDS
""")

print("\n\n2. TOP CANDIDATES FOR IMBALANCED FRAUD DETECTION (ROC-AUC > 0.90)")
print("-" * 130)

high_roc = all_results[all_results["ROC-AUC"] > 0.90].sort_values(
    ["Features", "ROC-AUC"], ascending=[True, False]
)
display_cols = ["Model", "Features", "ROC-AUC", "PR-AUC", "F1", "Recall", "Precision", "TP", "FP", "FN"]
print(high_roc[display_cols].head(15).to_string(index=False))

print("\n\n3. DETAILED COMPARISON - THREE MAIN CANDIDATES")
print("=" * 130)

focus = [
    "RF - Baseline",
    "RF - Feature Engineering",
    "RF - Reduced Baseline",
    "RF - Reduced Feature Engineering",
    "LightGBM - Reduced Feature Engineering",
    "XGBoost - Reduced Feature Engineering",
    "CatBoost - Reduced Feature Engineering",
]

for model_name in focus:
    row = all_results[all_results["Model"] == model_name]
    if len(row) == 0:
        continue
    row = row.iloc[0]

    print(f"\n{model_name}")
    print("-" * 130)

    print("\nCore Metrics (for imbalanced data):")
    print(f"  ROC-AUC:              {row['ROC-AUC']:.4f}  <- Main comparison metric")
    print(f"  PR-AUC:               {row['PR-AUC']:.4f}  <- Critical for fraud (rare events)")
    print(f"  F1 Score:             {row['F1']:.4f}   <- Balance precision & recall")

    print("\nFraud Detection Performance:")
    print(f"  Recall (catch rate):  {row['Recall']:.4f}   <- % of frauds actually caught")
    print(f"  Precision:            {row['Precision']:.4f}  <- % of alerts that are real frauds")

    print("\nConfusion Matrix Breakdown:")
    tn, fp, fn, tp = int(row["TN"]), int(row["FP"]), int(row["FN"]), int(row["TP"])
    total = tn + fp + fn + tp
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0

    print(f"  True Negatives (TN):   {tn:8d}  <- Legitimate txns correctly allowed")
    print(f"  False Positives (FP):  {fp:8d}  <- Legitimate txns falsely flagged (customer friction)")
    print(f"  False Negatives (FN):  {fn:8d}  <- Frauds missed (WORST - direct loss!)")
    print(f"  True Positives (TP):   {tp:8d}  <- Frauds caught (BEST)")
    print("  -----------------------------------")
    print(f"  Total samples:         {total:8d}")

    print("\nDerived Metrics:")
    print(f"  Specificity:          {specificity:.4f}   <- % of legitimate txns correctly allowed")
    print(f"  Sensitivity:          {sensitivity:.4f}   <- % of frauds caught (same as Recall)")
    print(f"  False Alarm Rate:     {fp / (fp + tn):.4f}   <- % of legitimate txns falsely flagged")
    print(f"  False Negative Rate:  {fn / (fn + tp):.4f}   <- % of frauds missed (minimize this!)")

    print(f"  Features:             {int(row['Features'])} columns")

print("\n\n4. RECOMMENDATION FOR IMBALANCED FRAUD DETECTION")
print("=" * 130)
print("""
KEY INSIGHT FOR IMBALANCED DATA:

Priority:
1. MINIMIZE FN (False Negatives / Missed Frauds) <- Direct business loss
2. MAINTAIN ROC-AUC > 0.90 <- Threshold-independent quality
3. MAXIMIZE F1 & Recall <- Better fraud detection
4. MANAGE FP (False Positives) <- Customer experience
5. IGNORE Accuracy <- Misleading for imbalanced data

BEST CHOICE: re-run 02_ml_models_fit.ipynb, then pick the row with highest PR-AUC / F1 among the four tables.

Compare Baseline vs Feature Engineering vs Reduced Baseline vs Reduced Feature Engineering
for each model family. Group-ablation permutations (Remove C / D / M / id / V) are no longer trained.
""")
print("=" * 130)


IMBALANCED DATASET ANALYSIS - FOR FRAUD DETECTION
Focus: ROC-AUC, PR-AUC, F1, Recall (NOT Accuracy - accuracy is misleading!)

1. KEY METRICS FOR IMBALANCED DATA
----------------------------------------------------------------------------------------------------------------------------------

For fraud detection (imbalanced dataset):

ROC-AUC (0.85-0.95 good)
  - Threshold-independent, handles class imbalance naturally
  - Main metric for comparing models

PR-AUC (Precision-Recall)
  - Especially important when fraud is rare
  - Shows trade-off: catch more frauds vs avoid false alerts

F1 Score
  - Harmonic mean of Precision & Recall
  - Balances both metrics

Recall (Sensitivity)
  - % of actual frauds caught
  - Missing a fraud = business loss

Precision
  - % of fraud alerts that are actually fraud
  - False alert = customer friction

Confusion Matrix (TN, FP, FN, TP)
  - FN (False Negatives) = frauds we missed - WORST outcome
  - FP (False Positives) = customers falsely flagged - c

## Feature-count trade-off

Best model at each width, and the smallest feature set that still clears ROC-AUC 0.90.

In [6]:
print("=" * 100)
print("ANALYZING ALL CONFIGURATIONS: Looking for Best Trade-offs")
print("=" * 100)

print("\n1. BY FEATURE COUNT (Least Parameters)")
print("-" * 100)

for feature_count in sorted(all_results["Features"].unique()):
    group = all_results[all_results["Features"] == feature_count].sort_values(
        "ROC-AUC", ascending=False
    )
    if len(group) > 0:
        best = group.iloc[0]
        print(
            f"\nFeatures: {int(feature_count):3d} | Best: {best['Model'][:40]:40s} | "
            f"ROC-AUC: {best['ROC-AUC']:.4f} | F1: {best['F1']:.4f}"
        )

print("\n\n2. TRADE-OFF ANALYSIS (Best Metrics vs Fewest Features)")
print("-" * 100)

sorted_by_features = all_results.sort_values(["Features", "ROC-AUC"], ascending=[True, False])

print("\nTop performer in each feature-reduction tier:")
seen_features = set()
count = 0
for _, row in sorted_by_features.iterrows():
    if row["Features"] not in seen_features and count < 8:
        seen_features.add(row["Features"])
        print(
            f"  {int(row['Features']):3d} features | {row['Model'][:45]:45s} | "
            f"ROC-AUC: {row['ROC-AUC']:.4f} | F1: {row['F1']:.4f} | Type: {row['ModelType']}"
        )
        count += 1

print("\n\n3. PER MODEL FAMILY (fewest features first)")
print("-" * 100)
for family in MODEL_ORDER:
    frame = all_results[all_results["ModelType"] == family]
    if frame.empty:
        continue
    print(f"\n{family}")
    display(
        frame.nsmallest(10, "Features")[
            [c for c in ["Model", "Features", "ROC-AUC", "PR-AUC", "F1", "Accuracy"] if c in frame.columns]
        ]
    )

print("\n\n4. BEST BY DIFFERENT METRICS")
print("-" * 100)

few = all_results[all_results["Features"] < 100]
if few.empty:
    print("No runs with <100 features (group ablations were removed). Ranking the four tables instead.")
    few = all_results
best_roc_few = few.nlargest(1, "ROC-AUC").iloc[0]
print("\nBest ROC-AUC:")
print(
    f"  {best_roc_few['Model']} | Features: {best_roc_few['Features']:.0f} | "
    f"ROC-AUC: {best_roc_few['ROC-AUC']:.4f} | F1: {best_roc_few['F1']:.4f}"
)
best_f1_few = few.nlargest(1, "F1").iloc[0]
print("\nBest F1:")
print(
    f"  {best_f1_few['Model']} | Features: {best_f1_few['Features']:.0f} | "
    f"ROC-AUC: {best_f1_few['ROC-AUC']:.4f} | F1: {best_f1_few['F1']:.4f}"
)

ranked = all_results.copy()
ranked["Balance"] = (ranked["ROC-AUC"] + ranked["F1"]) / 2
best_balanced_few = ranked.nlargest(1, "Balance").iloc[0]
print("\nBest Balance (ROC-AUC + F1 avg):")
print(
    f"  {best_balanced_few['Model']} | Features: {best_balanced_few['Features']:.0f} | "
    f"ROC-AUC: {best_balanced_few['ROC-AUC']:.4f} | F1: {best_balanced_few['F1']:.4f}"
)

qualifying = all_results[all_results["ROC-AUC"] > 0.90]
if len(qualifying) > 0:
    best_minimal = qualifying.nsmallest(1, "Features").iloc[0]
    print("\nMinimum features with ROC-AUC >0.90:")
    print(
        f"  {best_minimal['Model']} | Features: {best_minimal['Features']:.0f} | "
        f"ROC-AUC: {best_minimal['ROC-AUC']:.4f} | F1: {best_minimal['F1']:.4f}"
    )

print("\n\n" + "=" * 100)


ANALYZING ALL CONFIGURATIONS: Looking for Best Trade-offs

1. BY FEATURE COUNT (Least Parameters)
----------------------------------------------------------------------------------------------------

Features: 342 | Best: XGBoost - Reduced Baseline               | ROC-AUC: 0.9120 | F1: 0.5262

Features: 348 | Best: XGBoost - Reduced Feature Engineering    | ROC-AUC: 0.9163 | F1: 0.5536

Features: 432 | Best: XGBoost - Baseline                       | ROC-AUC: 0.9114 | F1: 0.5265

Features: 439 | Best: XGBoost - Feature Engineering            | ROC-AUC: 0.9196 | F1: 0.5813


2. TRADE-OFF ANALYSIS (Best Metrics vs Fewest Features)
----------------------------------------------------------------------------------------------------

Top performer in each feature-reduction tier:
  342 features | XGBoost - Reduced Baseline                    | ROC-AUC: 0.9120 | F1: 0.5262 | Type: XGBoost
  348 features | XGBoost - Reduced Feature Engineering         | ROC-AUC: 0.9163 | F1: 0.5536 | Type: XGB

,Model,Features,ROC-AUC,PR-AUC,F1,Accuracy
2,Logistic Regression - Reduced Baseline,342,0.8248,0.1771,0.1513,0.6881
3,Logistic Regression - Reduced Feature Engineering,348,0.8248,0.1802,0.1536,0.6976
0,Logistic Regression - Baseline,432,0.8335,0.2067,0.1583,0.7058
1,Logistic Regression - Feature Engineering,439,0.8334,0.2094,0.1595,0.7091



DecisionTree


,Model,Features,ROC-AUC,PR-AUC,F1,Accuracy
6,Decision Tree - Reduced Baseline,342,0.8221,0.3320,0.2044,0.8237
7,Decision Tree - Reduced Feature Engineering,348,0.8307,0.3506,0.2193,0.8306
4,Decision Tree - Baseline,432,0.8251,0.3310,0.1986,0.8150
5,Decision Tree - Feature Engineering,439,0.7965,0.3368,0.2035,0.8181



RandomForest


...


XGBoost


,Model,Features,ROC-AUC,PR-AUC,F1,Accuracy
18,XGBoost - Reduced Baseline,342,0.9120,0.5604,0.5262,0.9653
19,XGBoost - Reduced Feature Engineering,348,0.9163,0.5717,0.5536,0.9704
16,XGBoost - Baseline,432,0.9114,0.5624,0.5265,0.9647
17,XGBoost - Feature Engineering,439,0.9196,0.5965,0.5813,0.9749



CatBoost


,Model,Features,ROC-AUC,PR-AUC,F1,Accuracy
22,CatBoost - Reduced Baseline,342,0.9093,0.5119,0.3551,0.9099
23,CatBoost - Reduced Feature Engineering,348,0.9095,0.5094,0.3409,0.9034
20,CatBoost - Baseline,432,0.9078,0.5156,0.3420,0.9025
21,CatBoost - Feature Engineering,439,0.9087,0.5065,0.3323,0.8987




4. BEST BY DIFFERENT METRICS
----------------------------------------------------------------------------------------------------
No runs with <100 features (group ablations were removed). Ranking the four tables instead.

Best ROC-AUC:
  XGBoost - Feature Engineering | Features: 439 | ROC-AUC: 0.9196 | F1: 0.5813

Best F1:
  XGBoost - Feature Engineering | Features: 439 | ROC-AUC: 0.9196 | F1: 0.5813

Best Balance (ROC-AUC + F1 avg):
  XGBoost - Feature Engineering | Features: 439 | ROC-AUC: 0.9196 | F1: 0.5813

Minimum features with ROC-AUC >0.90:
  LightGBM - Reduced Baseline | Features: 342 | ROC-AUC: 0.9118 | F1: 0.5213




## Test submissions

Refit every Optuna run on the full train table for that feature set, then score `dataset/merged_test.parquet`. No GridSearchCV — params come from `BestParams` in `ml_results.parquet`.


In [7]:
RANDOM_SEED = 42
ID_COLS = ["isFraud", "TransactionID"]
FE_COLS = ["uid", "uid2", "DT_month", "DT_week", "DT_day", "DT_weekday", "DT_hour"]


def _scale_pos_weight(y):
    n_neg = int((y == 0).sum())
    n_pos = max(int((y == 1).sum()), 1)
    return n_neg / n_pos


def build_model(model_type, params, y_train, n_jobs=-1):
    p = dict(params)
    if model_type == "LogisticRegression":
        return Pipeline(
            [
                ("scaler", StandardScaler()),
                (
                    "clf",
                    LogisticRegression(
                        class_weight="balanced",
                        max_iter=2000,
                        solver="lbfgs",
                        random_state=RANDOM_SEED,
                        **p,
                    ),
                ),
            ]
        )
    if model_type == "DecisionTree":
        return DecisionTreeClassifier(
            class_weight="balanced", random_state=RANDOM_SEED, **p
        )
    if model_type == "RandomForest":
        return RandomForestClassifier(
            class_weight="balanced", random_state=RANDOM_SEED, n_jobs=n_jobs, **p
        )
    if model_type == "LightGBM":
        return LGBMClassifier(
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=n_jobs,
            verbosity=-1,
            subsample_freq=1,
            **p,
        )
    if model_type == "XGBoost":
        return XGBClassifier(
            random_state=RANDOM_SEED,
            n_jobs=n_jobs,
            eval_metric="logloss",
            tree_method="hist",
            scale_pos_weight=_scale_pos_weight(y_train),
            **p,
        )
    if model_type == "CatBoost":
        return CatBoostClassifier(
            random_seed=RANDOM_SEED,
            auto_class_weights="Balanced",
            verbose=False,
            allow_writing_files=False,
            thread_count=n_jobs if n_jobs > 0 else -1,
            **p,
        )
    raise ValueError(f"Unknown model_type {model_type}")


def parse_params(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return {}
    if isinstance(value, dict):
        return dict(value)
    return json.loads(value)


def feature_columns(table, dataset):
    if dataset in ("Baseline", "Reduced Baseline"):
        return [c for c in table.columns if c not in ID_COLS + FE_COLS]
    return [c for c in table.columns if c not in ID_COLS]


def submission_name(model_type, dataset):
    return f"{model_type}_{dataset.replace(' ', '_')}_submission.csv"


def fraud_score(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    return model.decision_function(X)


runs = all_results.copy()
runs = runs[~runs["Model"].astype(str).str.startswith(("SVM -", "KNN -"))]
if "BestParams" not in runs.columns or runs["BestParams"].isna().any():
    raise ValueError("BestParams missing — re-run Optuna cells in 02_ml_models_fit.ipynb")

print("Loading train / reduced / test …")
train_full = (
    pd.read_parquet(DATASET_PATH / "merged_train.parquet")
    .sort_values("TransactionDT")
    .reset_index(drop=True)
)
train_reduced = (
    pd.read_parquet(DATASET_PATH / "merged_train_reduced.parquet")
    .sort_values("TransactionDT")
    .reset_index(drop=True)
)
test = pd.read_parquet(DATASET_PATH / "merged_test.parquet")
test_ids = test["TransactionID"].to_numpy()
print(f"train {train_full.shape}  reduced {train_reduced.shape}  test {test.shape}")

written = []
for _, row in runs.sort_values(["ModelType", "Dataset"]).iterrows():
    model_type = str(row["ModelType"])
    dataset = str(row["Dataset"])
    out_path = RESULTS_DIR / submission_name(model_type, dataset)
    if out_path.exists():
        print(f"skip {out_path.name}")
        written.append(out_path.name)
        continue

    table = train_reduced if dataset.startswith("Reduced") else train_full
    cols = feature_columns(table, dataset)
    missing_cols = [c for c in cols if c not in test.columns]
    if missing_cols:
        raise ValueError(f"{row['Model']}: test missing {missing_cols[:8]}")

    params = parse_params(row["BestParams"])
    X_train = table[cols]
    y_train = table["isFraud"]
    X_test = test[cols]
    print(f"fit {row['Model']}  features={len(cols)}")
    model = build_model(model_type, params, y_train, n_jobs=-1)
    model.fit(X_train, y_train)
    scores = fraud_score(model, X_test)
    submission = pd.DataFrame(
        {"TransactionID": test_ids, "IsFraud": np.asarray(scores, dtype=np.float64)}
    )
    submission.to_csv(out_path, index=False)
    print(f"wrote {out_path.name}  rows={len(submission):,}")
    written.append(out_path.name)
    del model, X_train, X_test, scores, submission
    gc.collect()

print(f"\\n{len(written)} files in {RESULTS_DIR}")
for name in written:
    print(f"  {name}")


Loading train / reduced / test …
train (590540, 441)  reduced (590540, 350)  test (506691, 440)
skip CatBoost_Baseline_submission.csv
skip CatBoost_Feature_Engineering_submission.csv
skip CatBoost_Reduced_Baseline_submission.csv
skip CatBoost_Reduced_Feature_Engineering_submission.csv
skip DecisionTree_Baseline_submission.csv
skip DecisionTree_Feature_Engineering_submission.csv
skip DecisionTree_Reduced_Baseline_submission.csv
skip DecisionTree_Reduced_Feature_Engineering_submission.csv
skip LightGBM_Baseline_submission.csv
skip LightGBM_Feature_Engineering_submission.csv
skip LightGBM_Reduced_Baseline_submission.csv
skip LightGBM_Reduced_Feature_Engineering_submission.csv
skip LogisticRegression_Baseline_submission.csv
skip LogisticRegression_Feature_Engineering_submission.csv
skip LogisticRegression_Reduced_Baseline_submission.csv
skip LogisticRegression_Reduced_Feature_Engineering_submission.csv
skip RandomForest_Baseline_submission.csv
skip RandomForest_Feature_Engineering_submissi

## Summary

This notebook ranks the **six models × four feature tables** written by `02_ml_models_fit.ipynb`, then writes test submissions from those Optuna params to `results/`.
